# Notebook to analyse flood results

## Flood scenarios

### Clip city limits

In [32]:
import arcpy
import pandas as pd

aprx = arcpy.mp.ArcGISProject("CURRENT")
m = aprx.activeMap

# Scenario group layer
group_layer_name = "Flood_results_shape"
group_layer = next((lyr for lyr in m.listLayers() if lyr.isGroupLayer and lyr.name == group_layer_name), None)
if group_layer is None:
    raise Exception(f"Group layer '{group_layer_name}' not found.")

scenario_layers = [lyr for lyr in group_layer.listLayers() if lyr.isFeatureLayer]

# City boundary layer (from map)
boundary_layer_name = "Municipal_boundaries_Project"
boundary_lyr = next((lyr for lyr in m.listLayers() if lyr.isFeatureLayer and lyr.name == boundary_layer_name), None)
if boundary_lyr is None:
    raise Exception(f"Boundary layer '{boundary_layer_name}' not found in the map.")

print("Scenarios:", [l.name for l in scenario_layers])
print("Boundary layer:", boundary_lyr.name)


Scenarios: ['GDB_2yearflood_current', 'GDB_10yearflood_current', 'GDB_100yearflood_current', 'GDB_2yearflood_future', 'GDB_10yearflood_future', 'GDB_100yearflood_future']
Boundary layer: Municipal_boundaries_Project


In [33]:
# Where to store clipped outputs
out_gdb = aprx.defaultGeodatabase

results_city = []

for lyr in scenario_layers:
    out_fc = out_gdb + "\\" + f"{lyr.name}_CITY"

    # Clip polygons to city boundary
    arcpy.analysis.Clip(in_features=lyr, clip_features=boundary_lyr, out_feature_class=out_fc)

    # Sum area inside city
    total_area_ft2 = 0.0
    with arcpy.da.SearchCursor(out_fc, ["Shape_Area"]) as cursor:
        for (a,) in cursor:
            if a is not None:
                total_area_ft2 += float(a)

    results_city.append({
        "Scenario": lyr.name,
        "FloodExtentCity_ft2": total_area_ft2,
        "FloodExtentCity_acres": total_area_ft2 / 43560
    })

df_extent_city = pd.DataFrame(results_city).sort_values("Scenario").reset_index(drop=True)
df_extent_city


,Scenario,FloodExtentCity_ft2,FloodExtentCity_acres
0,GDB_100yearflood_current,3.260815e+07,748.580085
1,GDB_100yearflood_future,3.525054e+07,809.241100
2,GDB_10yearflood_current,1.732977e+07,397.836809
3,GDB_10yearflood_future,2.399425e+07,550.832191
4,GDB_2yearflood_current,8.112047e+06,186.226970
5,GDB_2yearflood_future,1.679160e+07,385.482015


### Compare current x future

In [34]:
# Separate current and future scenarios (CITY-LIMITED)
current = df_extent_city[~df_extent_city["Scenario"].str.contains("future")].copy()
future  = df_extent_city[df_extent_city["Scenario"].str.contains("future")].copy()

# Common key (return period)
current["RP"] = current["Scenario"].str.extract(r"(\d+year)")
future["RP"]  = future["Scenario"].str.extract(r"(\d+year)")

# Merge current and future
df_increase_city = current.merge(
    future,
    on="RP",
    suffixes=("_current", "_future")
)

# Calculate increase (acres)
df_increase_city["Increase_acres"] = (
    df_increase_city["FloodExtentCity_acres_future"]
    - df_increase_city["FloodExtentCity_acres_current"]
)

df_increase_city["Increase_percent"] = (
    df_increase_city["Increase_acres"]
    / df_increase_city["FloodExtentCity_acres_current"]
) * 100

# Keep only what matters for results
df_increase_city = df_increase_city[[
    "RP",
    "FloodExtentCity_acres_current",
    "FloodExtentCity_acres_future",
    "Increase_acres",
    "Increase_percent"
]]

df_increase_city


,RP,FloodExtentCity_acres_current,FloodExtentCity_acres_future,Increase_acres,Increase_percent
0,100year,748.580085,809.241100,60.661015,8.103477
1,10year,397.836809,550.832191,152.995382,38.456819
2,2year,186.226970,385.482015,199.255044,106.995804


In [35]:
print(current["Scenario"].tolist())
print(future["Scenario"].tolist())


['GDB_100yearflood_current', 'GDB_10yearflood_current', 'GDB_2yearflood_current']
['GDB_100yearflood_future', 'GDB_10yearflood_future', 'GDB_2yearflood_future']


### Compare to the city boundaries

In [36]:
# --- City area (acres) from Municipal_boundaries ---
city_area_ft2 = 0.0
with arcpy.da.SearchCursor(boundary_lyr, ["Shape_Area"]) as cursor:
    for (a,) in cursor:
        if a is not None:
            city_area_ft2 += float(a)

city_area_acres = city_area_ft2 / 43560
print("City area (acres):", city_area_acres)

# --- Add percent-of-city flooded to the city-limited flood extent table ---
df_city_share = df_extent_city.copy()
df_city_share["CityArea_acres"] = city_area_acres
df_city_share["PercentCityFlooded"] = (df_city_share["FloodExtentCity_acres"] / city_area_acres) * 100

# Results-ready view
df_city_share[["Scenario", "FloodExtentCity_acres", "PercentCityFlooded"]].sort_values("Scenario").reset_index(drop=True)


City area (acres): 2565.750418264262


,Scenario,FloodExtentCity_acres,PercentCityFlooded
0,GDB_100yearflood_current,748.580085,29.175873
1,GDB_100yearflood_future,809.241100,31.540133
2,GDB_10yearflood_current,397.836809,15.505671
3,GDB_10yearflood_future,550.832191,21.468658
4,GDB_2yearflood_current,186.226970,7.258187
5,GDB_2yearflood_future,385.482015,15.024143


## Roads intersection

In [49]:
import arcpy
import os
import pandas as pd

arcpy.env.overwriteOutput = True

# Names of layers in the map
FLOOD_GROUP_NAME = "Flood_results_shape_city"
ROADS_LAYER_NAME = "Road_PocomokeCity"
CITY_LAYER_NAME  = "Municipal_boundaries_Project"

# Access project and map
aprx = arcpy.mp.ArcGISProject("CURRENT")
m = aprx.activeMap
out_gdb = aprx.defaultGeodatabase
arcpy.env.addOutputsToMap = False

# Helper functions
def get_layer_by_name(map_obj, layer_name):
    lyr = map_obj.listLayers(layer_name)
    if not lyr:
        raise ValueError(f"Layer not found: {layer_name}")
    return lyr[0]

def add_length_ft(fc, field_name):
    if field_name not in [f.name for f in arcpy.ListFields(fc)]:
        arcpy.management.AddField(fc, field_name, "DOUBLE")
    arcpy.management.CalculateGeometryAttributes(
        fc, [[field_name, "LENGTH"]], length_unit="FEET_US"
    )

def sum_field(fc, field_name):
    total = 0.0
    with arcpy.da.SearchCursor(fc, [field_name]) as cur:
        for (v,) in cur:
            if v is not None:
                total += float(v)
    return total




In [50]:
# Step 1: Compute flooded roads per scenario (CITY)
flood_group = get_layer_by_name(m, FLOOD_GROUP_NAME)
roads_lyr   = get_layer_by_name(m, ROADS_LAYER_NAME)
city_lyr    = get_layer_by_name(m, CITY_LAYER_NAME)

roads_fc = roads_lyr.dataSource
city_fc  = city_lyr.dataSource

roads_city_fc = os.path.join(out_gdb, "Roads_CITY")
if arcpy.Exists(roads_city_fc):
    arcpy.management.Delete(roads_city_fc)

arcpy.analysis.Clip(roads_fc, city_fc, roads_city_fc)

add_length_ft(roads_city_fc, "Len_ft")
total_roads_len_ft = sum_field(roads_city_fc, "Len_ft")
total_roads_len_mi = total_roads_len_ft / 5280.0

print(f"Total road length inside city: {total_roads_len_mi:.2f} miles")

flood_layers = [lyr for lyr in flood_group.listLayers() if lyr.isFeatureLayer]
rows = []

for flood_lyr in flood_layers:
    scen = flood_lyr.name
    flood_fc = flood_lyr.dataSource

    out_fc = os.path.join(out_gdb, f"{scen}_RoadFlooded")
    if arcpy.Exists(out_fc):
        arcpy.management.Delete(out_fc)

    arcpy.analysis.Intersect(
        [roads_city_fc, flood_fc],
        out_fc,
        join_attributes="ALL",
        output_type="LINE"
    )

    add_length_ft(out_fc, "FloodLen_ft")
    flooded_len_ft = sum_field(out_fc, "FloodLen_ft")
    flooded_len_mi = flooded_len_ft / 5280.0
    pct_flooded = (flooded_len_ft / total_roads_len_ft) * 100

    rows.append([scen, flooded_len_mi, pct_flooded])

df_roads = pd.DataFrame(
    rows,
    columns=["Scenario", "FloodedRoadMiles", "PercentRoadsFlooded"]
).sort_values("Scenario").reset_index(drop=True)

df_roads



Total road length inside city: 45.79 miles


,Scenario,FloodedRoadMiles,PercentRoadsFlooded
0,GDB_100yearflood_current_CITY,9.090291,19.853024
1,GDB_100yearflood_future_CITY,9.905213,21.632797
2,GDB_10yearflood_current_CITY,3.821426,8.345922
3,GDB_10yearflood_future_CITY,4.534178,9.902558
4,GDB_2yearflood_current_CITY,1.673246,3.654338
5,GDB_2yearflood_future_CITY,1.923311,4.200475


In [51]:
# Step 2: Compare current vs future (roads)
current = df_roads[~df_roads["Scenario"].str.contains("future")].copy()
future  = df_roads[df_roads["Scenario"].str.contains("future")].copy()

current["RP"] = current["Scenario"].str.extract(r"(\d+year)")
future["RP"]  = future["Scenario"].str.extract(r"(\d+year)")

df_roads_compare = current.merge(
    future,
    on="RP",
    suffixes=("_current", "_future")
)

df_roads_compare["Increase_miles"] = (
    df_roads_compare["FloodedRoadMiles_future"]
    - df_roads_compare["FloodedRoadMiles_current"]
)

df_roads_compare["Increase_percent"] = (
    df_roads_compare["Increase_miles"]
    / df_roads_compare["FloodedRoadMiles_current"]
) * 100

df_roads_compare = df_roads_compare[[
    "RP",
    "FloodedRoadMiles_current",
    "FloodedRoadMiles_future",
    "Increase_miles",
    "Increase_percent"
]]

df_roads_compare

,RP,FloodedRoadMiles_current,FloodedRoadMiles_future,Increase_miles,Increase_percent
0,100year,9.090291,9.905213,0.814921,8.964745
1,10year,3.821426,4.534178,0.712751,18.651451
2,2year,1.673246,1.923311,0.250065,14.944895


## Buildings intersection
This approach counts a building as “flooded” if it intersects any flood polygon.

In [57]:
import arcpy
import os
import pandas as pd

arcpy.env.overwriteOutput = True
arcpy.env.addOutputsToMap = False

FLOOD_GROUP_NAME = "Flood_results_shape_city"
BUILDINGS_LAYER_NAME = "Buildings"  # <- change to your buildings layer name in Contents
CITY_LAYER_NAME  = "Municipal_boundaries_Project"

aprx = arcpy.mp.ArcGISProject("CURRENT")
m = aprx.activeMap
out_gdb = aprx.defaultGeodatabase

def get_layer_by_name(map_obj, layer_name):
    lyr = map_obj.listLayers(layer_name)
    if not lyr:
        raise ValueError(f"Layer not found: {layer_name}")
    return lyr[0]


In [58]:
#Step 1: Flooded buildings per scenario
flood_group = get_layer_by_name(m, FLOOD_GROUP_NAME)
bldg_lyr    = get_layer_by_name(m, BUILDINGS_LAYER_NAME)
city_lyr    = get_layer_by_name(m, CITY_LAYER_NAME)

bldg_fc = bldg_lyr.dataSource
city_fc = city_lyr.dataSource

bldg_city_fc = os.path.join(out_gdb, "Buildings_CITY")
if arcpy.Exists(bldg_city_fc):
    arcpy.management.Delete(bldg_city_fc)

arcpy.analysis.Clip(bldg_fc, city_fc, bldg_city_fc)

flood_layers = [lyr for lyr in flood_group.listLayers()
                if lyr.isFeatureLayer and lyr.name.endswith("_CITY") and "RoadFlooded" not in lyr.name]

rows = []

for flood_lyr in flood_layers:
    scen = flood_lyr.name
    flood_fc = flood_lyr.dataSource

    out_fc = os.path.join(out_gdb, f"{scen}_BuildingsFlooded")
    if arcpy.Exists(out_fc):
        arcpy.management.Delete(out_fc)

    arcpy.analysis.SpatialJoin(
        target_features=bldg_city_fc,
        join_features=flood_fc,
        out_feature_class=out_fc,
        join_operation="JOIN_ONE_TO_ONE",
        join_type="KEEP_COMMON",
        match_option="INTERSECT"
    )

    flooded_count = int(arcpy.management.GetCount(out_fc)[0])
    rows.append([scen, flooded_count])

df_buildings_counts = pd.DataFrame(
    rows,
    columns=["Scenario", "FloodedBuildings_count"]
).sort_values("Scenario").reset_index(drop=True)

df_buildings_counts


,Scenario,FloodedBuildings_count
0,GDB_100yearflood_current_CITY,256
1,GDB_100yearflood_future_CITY,284
2,GDB_10yearflood_current_CITY,90
3,GDB_10yearflood_future_CITY,114
4,GDB_2yearflood_current_CITY,28
5,GDB_2yearflood_future_CITY,43


In [59]:
current = df_buildings_counts[~df_buildings_counts["Scenario"].str.contains("future")].copy()
future  = df_buildings_counts[df_buildings_counts["Scenario"].str.contains("future")].copy()

current["RP"] = current["Scenario"].str.extract(r"(\d+year)")
future["RP"]  = future["Scenario"].str.extract(r"(\d+year)")

df_buildings_current_future = current.merge(
    future,
    on="RP",
    suffixes=("_current", "_future")
)

df_buildings_current_future["Increase_count"] = (
    df_buildings_current_future["FloodedBuildings_count_future"]
    - df_buildings_current_future["FloodedBuildings_count_current"]
)

df_buildings_current_future["Increase_percent"] = (
    df_buildings_current_future["Increase_count"]
    / df_buildings_current_future["FloodedBuildings_count_current"]
) * 100

df_buildings_current_future = df_buildings_current_future[[
    "RP",
    "FloodedBuildings_count_current",
    "FloodedBuildings_count_future",
    "Increase_count",
    "Increase_percent"
]].rename(columns={
    "FloodedBuildings_count_current": "FloodedBuildings_current",
    "FloodedBuildings_count_future": "FloodedBuildings_future"
})

df_buildings_current_future


,RP,FloodedBuildings_current,FloodedBuildings_future,Increase_count,Increase_percent
0,100year,256,284,28,10.937500
1,10year,90,114,24,26.666667
2,2year,28,43,15,53.571429


In [60]:
total_bldgs_city = int(arcpy.management.GetCount(bldg_city_fc)[0])

df_buildings_city_percent = df_buildings_counts.copy()
df_buildings_city_percent["TotalBuildings_city"] = total_bldgs_city
df_buildings_city_percent["PercentCityBuildingsFlooded"] = (
    df_buildings_city_percent["FloodedBuildings_count"] / total_bldgs_city
) * 100

df_buildings_city_percent


,Scenario,FloodedBuildings_count,TotalBuildings_city,PercentCityBuildingsFlooded
0,GDB_100yearflood_current_CITY,256,1638,15.628816
1,GDB_100yearflood_future_CITY,284,1638,17.338217
2,GDB_10yearflood_current_CITY,90,1638,5.494505
3,GDB_10yearflood_future_CITY,114,1638,6.959707
4,GDB_2yearflood_current_CITY,28,1638,1.709402
5,GDB_2yearflood_future_CITY,43,1638,2.625153
